# Module 3: Annotation queues and a self-improving eval

In Module 2 the reference labels came from analysts. But new transactions arrive every day, and nobody has
labelled them yet. In this module you close the loop:

![The improvement loop](../images/eval_loop.png)

1. Run the agent on new, unlabelled transactions, with tracing on.
2. Send its decisions to an **annotation queue** in LangSmith, where a human reviews them.
3. Read the reviewer's corrections back.
4. Add them to the eval dataset, so the eval covers more cases each time.
5. Build an LLM judge that learns from the reviewers' comments, so it grades the agent's reasoning the way your
   reviewers would.

You need `LANGSMITH_API_KEY` in `.env`, and you need to have run Module 2, which creates the dataset.

Step 3 happens in the LangSmith UI, so unlike Modules 1 and 2 this module has no matching `.py` file.

In [ ]:
# Let Python find `agent.py` and the `utils/`, `tools/` and `evals/` folders.
# This notebook lives in modules/, so the workshop root is one level up.
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
from langsmith import Client

from utils.attendee import ATTENDEE
from scripts.run_experiments import DATASET_NAME

client = Client()
assert client.has_dataset(dataset_name=DATASET_NAME), "Run Module 2 first: it creates the dataset."

## Step 1: Run the agent on new transactions (3 min)

Here is Priya's next week of transactions, from the `transactions` table. Nobody has labelled them.

Each `invoke` creates a trace in LangSmith. We give each one our own **run ID**, so we can find it again and add
it to the queue. Reviews will happen in the queue, so if the agent asks for help we accept its own decision for
now.

In [ ]:
import uuid

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.types import Command
from langsmith import tracing_context

from agent import build_agent
from utils.state import OCT_19, MyAgentContext
from utils.transaction_processing import transaction_stream, transaction_message, decision_log

NEW_TRANSACTIONS = transaction_stream("USR-001", start=OCT_19)

agent = build_agent(checkpointer=InMemorySaver(), store=InMemoryStore())
config = {"configurable": {"thread_id": "USR-001-week-2"}}
run_ids = {}  # transaction ID -> LangSmith run ID

with tracing_context(enabled=True):
    for tx in NEW_TRANSACTIONS:
        context = MyAgentContext(user_id="USR-001")
        run_ids[tx["transaction_id"]] = uuid.uuid4()
        run_config = {**config, "run_id": run_ids[tx["transaction_id"]], "run_name": f"Assess {tx['transaction_id']}"}
        result = agent.invoke(transaction_message(tx["transaction_id"]), run_config, context=context)
        if "__interrupt__" in result:
            question = result["__interrupt__"][0].value
            result = agent.invoke(Command(resume={"decision": question["agent_decision"]}), config, context=context)
        print(f"{tx['transaction_id']}: {result['messages'][-1].text}")

decisions = {decision["transaction_id"]: decision for decision in decision_log(agent, "USR-001")}

## Step 2: Create an annotation queue (2 min)

An annotation queue is a to-do list of runs for humans to review. Its **rubric** tells reviewers what to do and
which feedback to leave. We ask for one piece of feedback, `correct_risk_score`: the score the reviewer would
have given. We register it as a number from 0 to 100, so the UI shows the right input.

The queue's name ends with your `ATTENDEE_NAME`. If you run this cell again, it reuses your queue.

In [ ]:
from langchain_core.tracers.langchain import wait_for_all_tracers

wait_for_all_tracers()  # make sure the runs have reached LangSmith before we queue them

try:
    client.create_feedback_config("correct_risk_score", feedback_config={"type": "continuous", "min": 0, "max": 100})
except Exception:
    pass  # it already exists in this workspace

QUEUE_NAME = f"transaction-monitoring-reviews-{ATTENDEE}"
existing = list(client.list_annotation_queues(name=QUEUE_NAME))
queue = existing[0] if existing else client.create_annotation_queue(
    name=QUEUE_NAME,
    description="Check the transaction-monitoring agent's risk scores.",
    rubric_instructions=(
        "Open the record_decision tool call to see the agent's score and reason. "
        "Add 'correct_risk_score': the score from 0 to 100 that you would give. "
        "Use the comment to explain why."
    ),
    rubric_items=[{"feedback_key": "correct_risk_score", "description": "The risk score (0-100) you would give."}],
)
client.add_runs_to_annotation_queue(queue.id, run_ids=list(run_ids.values()))
print(f"Queue '{queue.name}' has {len(run_ids)} runs to review.")

## Step 3: Review in LangSmith (5 min)

1. In LangSmith, open **Annotation queues** and select the queue printed above.
2. For each run, open the `record_decision` call to see the agent's score and reason.
3. Add `correct_risk_score` with the score you'd give, and a comment explaining why.
4. Select **Done** to move to the next run.

**Short on time?** Set `SIMULATE_ANNOTATIONS = True` and run the cell below. It leaves the reviews from
the `simulated_annotations` table in `data/` for you. Don't do both, or each run will have two reviews.

In [ ]:
from utils import db

SIMULATE_ANNOTATIONS = False

if SIMULATE_ANNOTATIONS:
    for tx_id, run_id in run_ids.items():
        annotation = db.get("simulated_annotations", transaction_id=tx_id)
        client.create_feedback(
            run_id, key="correct_risk_score", score=annotation["correct_risk_score"], comment=annotation["comment"]
        )
    print(f"Added {len(run_ids)} simulated reviews.")

## Step 4: Read the corrections back (2 min)

Every review is stored as **feedback** on its run. We read it back and compare it with the agent's scores.

In [ ]:
transaction_for_run = {str(run_id): tx_id for tx_id, run_id in run_ids.items()}
corrections = {}

for feedback in client.list_feedback(run_ids=list(run_ids.values()), feedback_key=["correct_risk_score"]):
    tx_id = transaction_for_run[str(feedback.run_id)]
    corrections[tx_id] = {"score": feedback.score, "comment": feedback.comment or ""}

if not corrections:
    print("No reviews yet. Review the runs in the queue (or simulate them), then run this cell again.")
for tx_id, correction in corrections.items():
    print(f"{tx_id}: agent {decisions[tx_id]['risk_score']:>3}, reviewer {correction['score']:>5.0f}  {correction['comment']}")

## Step 5: Grow the eval dataset (2 min)

The corrections become reference labels for a new example in the Module 2 dataset. We turn each score into a
label with the same rules the agent uses: **block** from 70 up, and **ambiguous** (ask a human) from 30 to 69.

Every review round adds real, reviewed cases to the dataset, so the eval keeps up with what the agent actually
sees. Run this cell once per review round: each run adds another example.

In [ ]:
def label_from_score(score: float) -> dict:
    return {"decision": "block" if score >= 70 else "approve", "risk_score": score, "ambiguous": 30 <= score < 70}

reviewed = [tx for tx in NEW_TRANSACTIONS if tx["transaction_id"] in corrections]
labels = {tx["transaction_id"]: label_from_score(corrections[tx["transaction_id"]]["score"]) for tx in reviewed}

if reviewed:
    client.create_examples(
        dataset_name=DATASET_NAME,
        examples=[
            {
                "inputs": {"user_id": "USR-001", "transactions": reviewed, "reviewer_answers": {}},
                "outputs": {"labels": labels},
            }
        ],
    )
    print(f"Added an example with {len(reviewed)} reviewed transactions to '{DATASET_NAME}'.")
else:
    print("No reviews yet, so nothing was added. Go back to step 3, then run steps 4 and 5 again.")

## Step 6: A judge that learns from your reviewers (5 min)

The Module 2 evaluators check numbers. They can't tell whether the agent's **reasons** make sense. An LLM judge
can, but it should judge the way your reviewers do, not the way the model happens to.

So we show the judge the reviewers' corrections as worked examples. We count the agent's reasoning as sound when
the reviewer's score was within 20 points of the agent's. Every time reviewers annotate more runs, the judge
gets more examples. That is what makes the eval **self-improving**.

In [ ]:
from pydantic import BaseModel, Field

from langchain.chat_models import init_chat_model
from utils.models import model

class Verdict(BaseModel):
    sound: bool = Field(description="True if the reasoning is sound and the score is in the right range")
    explanation: str = Field(description="One sentence explaining the verdict")

judge = init_chat_model(model).with_structured_output(Verdict, method="json_schema")

def reviewer_examples() -> str:
    """Turn the reviewers' corrections into worked examples for the judge."""
    examples = []
    for tx_id, correction in corrections.items():
        agent_decision = decisions[tx_id]
        sound = abs(agent_decision["risk_score"] - correction["score"]) <= 20
        examples.append(
            f"- The agent scored {agent_decision['risk_score']} because: {agent_decision['reason']}\n"
            f"  The reviewer scored {correction['score']:.0f}: {correction['comment']}\n"
            f"  Verdict: {'sound' if sound else 'not sound'}"
        )
    return "\n".join(examples)


JUDGE_PROMPT = """You check a fraud agent's reasoning about card transactions.

This is how our human reviewers judged some of the agent's past decisions:
{examples}

Judge this decision the same way.
Transaction: {transaction}
Agent's score: {risk_score}
Agent's reason: {reason}"""


def reasoning_judge(inputs: dict, outputs: dict) -> float:
    """Share of the agent's decisions whose reasoning the judge rates as sound."""
    transactions = {tx["transaction_id"]: tx for tx in inputs["transactions"]}
    examples = reviewer_examples()
    verdicts = [
        judge.invoke(
            JUDGE_PROMPT.format(
                examples=examples,
                transaction=transactions.get(decision["transaction_id"]),
                risk_score=decision["risk_score"],
                reason=decision["reason"],
            )
        ).sound
        for decision in outputs["decisions"]
    ]
    return sum(verdicts) / len(verdicts) if verdicts else 0.0

Try the judge on one of this week's decisions:

In [ ]:
tx = NEW_TRANSACTIONS[2]
verdict = judge.invoke(
    JUDGE_PROMPT.format(
        examples=reviewer_examples(),
        transaction=tx,
        risk_score=decisions[tx["transaction_id"]]["risk_score"],
        reason=decisions[tx["transaction_id"]]["reason"],
    )
)
print(verdict)

## Step 7: Run the improved eval (3 min)

Run the evaluation again, on the bigger dataset and with the new judge. In LangSmith, compare this experiment
with your Module 2 baseline: there's one more example, and a new `reasoning_judge` score.

In [ ]:
from langsmith import evaluate

from evals.scoring_evals import score_alignment_eval, decision_accuracy_eval, review_routing_eval
from scripts.run_experiments import run_agent_on_stream, average_scores

EVALUATORS = [score_alignment_eval, decision_accuracy_eval, review_routing_eval, reasoning_judge]

def run_experiment(prefix: str, context_settings: MyAgentContext) -> dict:
    """Run the evaluation once and return the average scores. The full experiment is saved to LangSmith."""
    
    # `evaluate` calls the target with just the example's inputs, so bind the settings here.
    def target(inputs: dict) -> dict:
        return run_agent_on_stream(context_settings, inputs)

    results = evaluate(target, data=DATASET_NAME, evaluators=EVALUATORS, experiment_prefix=prefix, max_concurrency=2)
    
    return average_scores(results)


print("with-reviewer-feedback:", run_experiment(prefix="with-reviewer-feedback", context_settings=MyAgentContext()))

## Done!

You've built the full loop: the agent runs, humans review what it did, and their reviews improve both the
dataset and the judge that grade the next version of the agent.

## Stretch goals

1. **Queue only what matters.** Add only low-confidence runs, or runs where the judge disagrees with the agent.
2. **Automate it.** Set up a LangSmith automation rule that sends low-confidence production runs to the queue.
3. **Check the judge.** Measure how often the judge agrees with the reviewers, before and after the examples.